# 01 — Exploratory Data Analysis
**Project FORESIGHT — NorthBay Living**

This notebook produces the charts and insights for **Deliverable D2**  
(Data Quality & EDA Insight Memo).

Run all cells top-to-bottom **after** running `python -m src.pipeline`.

Charts are saved to `reports/` for inclusion in the memo.

## Section 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Make the repo root importable from the notebooks/ folder
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.config import PANEL_PATH

panel = pd.read_parquet(PANEL_PATH)
panel['date'] = pd.to_datetime(panel['date'])

print(f'Panel shape: {panel.shape}')
print()
print(panel.dtypes)

## Section 2 — Data Quality Summary

In [ ]:
# Null counts per column
print('Nulls per column:')
print(panel.isnull().sum())

# Unique SKUs and date range
print(f'\nUnique SKUs: {panel["sku_id"].nunique()}')
print(f'Date range : {panel["date"].min().date()} to {panel["date"].max().date()}')

# Category distribution
print('\nSKUs per canonical category:')
print(panel.groupby('category')['sku_id'].nunique())

## Section 3 — Demand Distribution

In [ ]:
import os
os.makedirs('../reports', exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 4))
panel['units_sold'].clip(upper=200).hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Daily units_sold (clipped at 200)')
ax.set_xlabel('Units Sold per Day')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/eda_demand_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/eda_demand_distribution.png')

## Section 4 — Top and Bottom SKUs

In [ ]:
sku_revenue = panel.groupby('sku_id')['revenue'].sum().sort_values(ascending=False)

print('Top 10 SKUs by total revenue:')
print(sku_revenue.head(10).to_string())

print('\nBottom 10 SKUs (dead-stock candidates):')
print(sku_revenue.tail(10).to_string())

## Section 5 — Seasonality

In [ ]:
# Day-of-week pattern
panel['dayofweek'] = panel['date'].dt.dayofweek
dow_avg = panel.groupby('dayofweek')['units_sold'].mean()
dow_avg.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(8, 4))
dow_avg.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average Daily Demand by Day of Week')
ax.set_ylabel('Avg units_sold')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../reports/eda_dow_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

# Monthly pattern (yearly seasonality)
monthly_avg = panel.groupby('month')['units_sold'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
monthly_avg.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Average Daily Demand by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Avg units_sold')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../reports/eda_monthly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

print('Charts saved to reports/')

## Section 6 — Promo Lift

In [ ]:
promo_comparison = panel.groupby('promo_flag')['units_sold'].mean()
print('Average daily demand by promo_flag:')
print(promo_comparison)

if 0 in promo_comparison.index and 1 in promo_comparison.index:
    lift_pct = (promo_comparison[1] / promo_comparison[0] - 1) * 100
    print(f'\nPromo lift: +{lift_pct:.1f}% average demand during promo periods')
    print('=> Record this in the D2 memo Section 2.3')

## Section 7 — Dead Stock Candidates

In [ ]:
# SKUs with high on-hand stock but very low recent sales
recent = panel[panel['date'] >= panel['date'].max() - pd.Timedelta(days=30)]
recent_sales  = recent.groupby('sku_id')['units_sold'].sum()
current_stock = panel.sort_values('date').groupby('sku_id')['on_hand_units'].last()

dead_stock = pd.DataFrame({
    'recent_30d_sales': recent_sales,
    'on_hand_units':    current_stock,
}).dropna()

dead_stock['days_cover'] = (
    dead_stock['on_hand_units'] / (dead_stock['recent_30d_sales'] / 30 + 0.001)
)

print('Potential dead stock (>90 days cover, low sales):')
candidates = dead_stock[dead_stock['days_cover'] > 90].sort_values('on_hand_units', ascending=False)
print(candidates.head(10).to_string())
print(f'\nTotal dead-stock candidates: {len(candidates)} SKUs')
print('=> Record count and capital estimate in D2 memo Section 3 Insight 3')

## Section 8 — Business Insights for D2 Memo

Fill in the placeholders below with actual numbers from Sections 5, 6, and 7 above,
then copy these three bullets into `reports/data_quality_eda_memo.md` Section 3.

---

**Insight 1 — Festive Season Demand Spike:**  
October–December drives approximately **[fill from Section 5 monthly chart]%**  
higher average daily demand vs the rest of the year. Current reorder points  
do not account for this seasonal lift, which likely explains repeat stockouts  
during the Diwali and Year-End Sale windows.

**Insight 2 — Promo Lift Quantified:**  
Promotional periods drive approximately **+[fill from Section 6]%** average  
daily demand vs non-promo periods. SKUs with lead times >14 days need  
replenishment orders placed at least 3 weeks before each promo window.

**Insight 3 — Dead Stock Risk:**  
**[fill from Section 7]** SKUs have >90 days of inventory cover at their  
current sales pace. These are candidates for markdown or clearance promotions  
to free working capital.